In [48]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import pandas as pd
import matplotlib.pyplot as plt

class Config:
    """Configuration class for model and training parameters."""
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    VGG_LAYERS = {
        '3': "relu1_1",
        '8': "relu2_1",
        '17': "relu3_1",
        '26': "relu4_1"
    }
    IMAGE_MEAN = [0.485, 0.456, 0.406]
    IMAGE_STD = [0.229, 0.224, 0.225]
    
    # Training parameters
    CONTENT_DIR = "D:\phd\\2 term 2\Deep Learning\\final proj\dataset\\train\images"
    STYLE_DIR = "D:\phd\\2 term 2\Deep Learning\\final proj\extra"
    LR = 1e-4
    STYLE_WEIGHT = 10.0
    EPOCHS = 200
    BATCH_SIZE = 8
    IMAGE_SIZE = 256
    LOG_INTERVAL = 100
    SAVE_INTERVAL = 1
    
    # Stylization parameters
    CONTENT_IMAGE = "../parts/1 image generation/0 content image.png"
    STYLE_IMAGE = "../parts/1 image generation/0 style image.png"
    DECODER_PATH = "decoder.pth"
    OUTPUT_IMAGE = "output.jpg"
    CONTENT_SIZE = 512
    STYLE_SIZE = 512
    ALPHA = 1.0

class VGGEncoder(nn.Module):
    """VGG-19 Encoder for feature extraction up to relu4_1."""
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 64, (3, 3), padding=1),  # conv1_1
            nn.ReLU(),  # relu1_1
            nn.Conv2d(64, 64, (3, 3), padding=1),  # conv1_2
            nn.ReLU(),  # relu1_2
            nn.MaxPool2d((2, 2), (2, 2), (0, 0), ceil_mode=True),
            nn.Conv2d(64, 128, (3, 3), padding=1),  # conv2_1
            nn.ReLU(),  # relu2_1
            nn.Conv2d(128, 128, (3, 3), padding=1),  # conv2_2
            nn.ReLU(),  # relu2_2
            nn.MaxPool2d((2, 2), (2, 2), (0, 0), ceil_mode=True),
            nn.Conv2d(128, 256, (3, 3), padding=1),  # conv3_1
            nn.ReLU(),  # relu3_1
            nn.Conv2d(256, 256, (3, 3), padding=1),  # conv3_2
            nn.ReLU(),  # relu3_2
            nn.Conv2d(256, 256, (3, 3), padding=1),  # conv3_3
            nn.ReLU(),  # relu3_3
            nn.Conv2d(256, 256, (3, 3), padding=1),  # conv3_4
            nn.ReLU(),  # relu3_4
            nn.MaxPool2d((2, 2), (2, 2), (0, 0), ceil_mode=True),
            nn.Conv2d(256, 512, (3, 3), padding=1),  # conv4_1
            nn.ReLU()   # relu4_1
        )
    
    def forward(self, x):
        return self.encoder(x)

class Decoder(nn.Module):
    """Decoder network to reconstruct images from stylized features."""
    def __init__(self):
        super().__init__()
        self.decoder = nn.Sequential(
            nn.ReflectionPad2d((1, 1, 1, 1)),
            nn.Conv2d(512, 256, (3, 3)),
            nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.ReflectionPad2d((1, 1, 1, 1)),
            nn.Conv2d(256, 256, (3, 3)),
            nn.ReLU(),
            nn.ReflectionPad2d((1, 1, 1, 1)),
            nn.Conv2d(256, 256, (3, 3)),
            nn.ReLU(),
            nn.ReflectionPad2d((1, 1, 1, 1)),
            nn.Conv2d(256, 256, (3, 3)),
            nn.ReLU(),
            nn.ReflectionPad2d((1, 1, 1, 1)),
            nn.Conv2d(256, 128, (3, 3)),
            nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.ReflectionPad2d((1, 1, 1, 1)),
            nn.Conv2d(128, 128, (3, 3)),
            nn.ReLU(),
            nn.ReflectionPad2d((1, 1, 1, 1)),
            nn.Conv2d(128, 64, (3, 3)),
            nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.ReflectionPad2d((1, 1, 1, 1)),
            nn.Conv2d(64, 64, (3, 3)),
            nn.ReLU(),
            nn.ReflectionPad2d((1, 1, 1, 1)),
            nn.Conv2d(64, 3, (3, 3)),
        )
    
    def forward(self, x):
        return self.decoder(x)

class LossNetwork(nn.Module):
    """VGG-19 based loss network for style and content loss computation."""
    def __init__(self):
        super().__init__()
        vgg = models.vgg19(pretrained=True).features
        self.vgg_layers = vgg
        self.layer_name_mapping = Config.VGG_LAYERS
    
    def forward(self, x):
        output = {}
        for name, module in self.vgg_layers._modules.items():
            x = module(x)
            if name in self.layer_name_mapping:
                output[self.layer_name_mapping[name]] = x
        return output

class AdaIN:
    """Adaptive Instance Normalization implementation."""
    @staticmethod
    def calc_mean_std(feat, eps=1e-5):
        size = feat.size()
        if len(size) != 4:
            raise ValueError("Expected 4D tensor for feature input")
        N, C = size[:2]
        feat_var = feat.view(N, C, -1).var(dim=2) + eps
        feat_std = feat_var.sqrt().view(N, C, 1, 1)
        feat_mean = feat.view(N, C, -1).mean(dim=2).view(N, C, 1, 1)
        return feat_mean, feat_std
    
    @staticmethod
    def adain(content_feat, style_feat):
        if content_feat.size()[:2] != style_feat.size()[:2]:
            raise ValueError(f"Content and style features must have matching batch and channel dimensions. "
                             f"Got content: {content_feat.size()[:2]}, style: {style_feat.size()[:2]}")
        size = content_feat.size()
        style_mean, style_std = AdaIN.calc_mean_std(style_feat)
        content_mean, content_std = AdaIN.calc_mean_std(content_feat)
        normalized_feat = (content_feat - content_mean.expand(size)) / content_std.expand(size)
        return normalized_feat * style_std.expand(size) + style_mean.expand(size)

class ImageProcessor:
    """Handles image transformations and loading."""
    @staticmethod
    def get_transform(size):
        return transforms.Compose([
            transforms.Resize(size),
            transforms.CenterCrop(size),
            transforms.ToTensor(),
            transforms.Normalize(mean=Config.IMAGE_MEAN, std=Config.IMAGE_STD)
        ])
    
    @staticmethod
    def inverse_normalize():
        return transforms.Normalize(
            mean=[-m/s for m, s in zip(Config.IMAGE_MEAN, Config.IMAGE_STD)],
            std=[1/s for s in Config.IMAGE_STD]
        )
    
    @staticmethod
    def load_image(image_path, transform):
        try:
            img = Image.open(image_path).convert('RGB')
            return transform(img).unsqueeze(0)
        except (IOError, OSError) as e:
            raise RuntimeError(f"Failed to load image {image_path}: {str(e)}")

class ImageDataset(Dataset):
    """Dataset for loading images from a directory."""
    def __init__(self, dir_path, transform):
        super().__init__()
        self.dir_path = dir_path
        self.transform = transform
        self.image_files = [f for f in os.listdir(dir_path) if os.path.isfile(os.path.join(dir_path, f))]
        if not self.image_files:
            raise ValueError(f"No valid images found in directory: {dir_path}")

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.dir_path, self.image_files[idx])
        try:
            return ImageProcessor.load_image(img_path, self.transform).squeeze(0)
        except RuntimeError:
            # Fallback to next image
            return self.__getitem__((idx + 1) % len(self.image_files))

class StyleTransferModel:
    """Main class for training and stylizing images using AdaIN."""
    def __init__(self):
        self.config = Config()
        self.encoder = VGGEncoder().to(Config.DEVICE)
        self.decoder = Decoder().to(Config.DEVICE)
        self.loss_net = LossNetwork().to(Config.DEVICE)
        self.content_losses = []
        self.style_losses = []
        self.total_losses = []
        self.iterations = []
        self.epochs = []
    
    def load_pretrained_weights(self):
        """Load pretrained VGG weights for encoder and loss network."""
        vgg_weights = models.vgg19(pretrained=True).features.state_dict()
        # Adjust state dictionary keys to match VGGEncoder structure
        adjusted_weights = {}
        for key, value in vgg_weights.items():
            new_key = f"encoder.{key}"
            adjusted_weights[new_key] = value
        # Load only the matching weights up to relu4_1
        encoder_state_dict = self.encoder.state_dict()
        encoder_state_dict.update({k: v for k, v in adjusted_weights.items() if k in encoder_state_dict})
        self.encoder.load_state_dict(encoder_state_dict)
        # Trim encoder to relu4_1 (21 layers)
        self.encoder = nn.Sequential(*list(self.encoder.encoder.children())[:21])
        for param in self.encoder.parameters():
            param.requires_grad = False
        for param in self.loss_net.parameters():
            param.requires_grad = False
    
    def compute_losses(self, gen_img_features, style_img_features, t):
        """Compute content and style losses."""
        content_loss = nn.functional.mse_loss(gen_img_features['relu4_1'], t)
        style_loss = 0
        for name, out_feat in gen_img_features.items():
            style_feat = style_img_features[name]
            out_mean, out_std = AdaIN.calc_mean_std(out_feat)
            style_mean, style_std = AdaIN.calc_mean_std(style_feat)
            style_loss += nn.functional.mse_loss(out_mean, style_mean) + \
                         nn.functional.mse_loss(out_std, style_std)
        return content_loss, style_loss


    def save_and_plot_losses(self, output_folder):
        """Save losses to CSV and plot combined and separate loss curves in a specified folder."""
        
        # Ensure the output directory exists
        os.makedirs(output_folder, exist_ok=True)

        # --- Save losses to CSV ---
        loss_data = {
            'Iteration': self.iterations,
            'Epoch': self.epochs,
            'Content_Loss': self.content_losses,
            'Style_Loss': self.style_losses,
            'Total_Loss': self.total_losses
        }
        df = pd.DataFrame(loss_data)
        csv_path = os.path.join(output_folder, 'losses.csv')
        df.to_csv(csv_path, index=False)
        print(f"Saved losses to {csv_path}")

        # --- Plot combined losses ---
        plt.figure(figsize=(10, 6))
        plt.plot(self.iterations, self.content_losses, label='Content Loss', color='blue')
        plt.plot(self.iterations, self.style_losses, label='Style Loss', color='orange')
        plt.plot(self.iterations, self.total_losses, label='Total Loss', color='green')
        plt.xlabel('Iteration')
        plt.ylabel('Loss')
        plt.title('Training Losses')
        plt.legend()
        plt.grid(True)
        combined_plot_path = os.path.join(output_folder, 'losses_combined.png')
        plt.savefig(combined_plot_path)
        plt.close()
        print(f"Saved combined loss plot to {combined_plot_path}")

        # --- Plot content loss separately ---
        plt.figure(figsize=(10, 6))
        plt.plot(self.iterations, self.content_losses, label='Content Loss', color='blue')
        plt.xlabel('Iteration')
        plt.ylabel('Content Loss')
        plt.title('Content Loss During Training')
        plt.legend()
        plt.grid(True)
        content_plot_path = os.path.join(output_folder, 'content_loss.png')
        plt.savefig(content_plot_path)
        plt.close()
        print(f"Saved content loss plot to {content_plot_path}")

        # --- Plot style loss separately ---
        plt.figure(figsize=(10, 6))
        plt.plot(self.iterations, self.style_losses, label='Style Loss', color='orange')
        plt.xlabel('Iteration')
        plt.ylabel('Style Loss')
        plt.title('Style Loss During Training')
        plt.legend()
        plt.grid(True)
        style_plot_path = os.path.join(output_folder, 'style_loss.png')
        plt.savefig(style_plot_path)
        plt.close()
        print(f"Saved style loss plot to {style_plot_path}")

    def train(self):
        """Training loop for the style transfer model."""
        self.load_pretrained_weights()
        optimizer = torch.optim.Adam(self.decoder.parameters(), lr=self.config.LR)
        
        content_dataset = ImageDataset(self.config.CONTENT_DIR, ImageProcessor.get_transform(self.config.IMAGE_SIZE))
        style_dataset = ImageDataset(self.config.STYLE_DIR, ImageProcessor.get_transform(self.config.IMAGE_SIZE))
        
        # Set drop_last=True to ensure equal batch sizes
        content_loader = DataLoader(content_dataset, batch_size=self.config.BATCH_SIZE, shuffle=True, drop_last=True)
        style_loader = DataLoader(style_dataset, batch_size=self.config.BATCH_SIZE, shuffle=True, drop_last=True)

        print(f"Content dataset size: {len(content_dataset)}, Style dataset size: {len(style_dataset)}")
        print("Starting training...")
        iteration = 0
        for epoch in range(self.config.EPOCHS):
            for i, (content_images, style_images) in enumerate(zip(content_loader, style_loader)):
                content_images = content_images.to(Config.DEVICE)
                style_images = style_images.to(Config.DEVICE)

                # Debug shapes
                content_features = self.encoder(content_images)
                style_features = self.encoder(style_images)
                # print(f"Epoch {epoch+1}, Step {i+1}: Content features shape: {content_features.shape}, "
                #       f"Style features shape: {style_features.shape}")

                t = AdaIN.adain(content_features, style_features)
                generated_images = self.decoder(t)

                gen_img_features = self.loss_net(generated_images)
                style_img_features = self.loss_net(style_images)
                
                c_loss, s_loss = self.compute_losses(gen_img_features, style_img_features, t)
                total_loss = self.config.STYLE_WEIGHT * s_loss + c_loss

                # Track losses
                self.content_losses.append(c_loss.item())
                self.style_losses.append(s_loss.item())
                self.total_losses.append(total_loss.item())
                self.iterations.append(iteration)
                self.epochs.append(epoch + 1)
                iteration += 1

                optimizer.zero_grad()
                total_loss.backward()
                optimizer.step()

                # if (i + 1) % self.config.LOG_INTERVAL == 0:
                if 1:
                    print(f"Epoch [{epoch+1}/{self.config.EPOCHS}], Step [{i+1}/{len(content_loader)}], "
                          f"Content Loss: {c_loss.item():.4f}, Style Loss: {s_loss.item():.4f}, "
                          f"Total Loss: {total_loss.item():.4f}")
                    
            if (epoch + 1) % self.config.SAVE_INTERVAL == 0:
                torch.save(self.decoder.state_dict(), f'decoder\decoder_epoch_{epoch+1}.pth')
                print(f"Saved decoder weights at epoch {epoch+1}")

        # Save and plot losses after training
        self.save_and_plot_losses("ada")

    def stylize(self):
        """Apply style transfer to a single image."""
        self.load_pretrained_weights()
        self.decoder.load_state_dict(torch.load(self.config.DECODER_PATH))
        self.encoder.eval()
        self.decoder.eval()

        content_tensor = ImageProcessor.load_image(self.config.CONTENT_IMAGE, 
                                                  ImageProcessor.get_transform(self.config.CONTENT_SIZE)).to(Config.DEVICE)
        style_tensor = ImageProcessor.load_image(self.config.STYLE_IMAGE, 
                                                ImageProcessor.get_transform(self.config.STYLE_SIZE)).to(Config.DEVICE)

        with torch.no_grad():
            content_feat = self.encoder(content_tensor)
            style_feat = self.encoder(style_tensor)
            print(f"Stylize: Content features shape: {content_feat.shape}, Style features shape: {style_feat.shape}")
            t = AdaIN.adain(content_feat, style_feat)
            t = self.config.ALPHA * t + (1 - self.config.ALPHA) * content_feat
            output_tensor = self.decoder(t)

        output_tensor = ImageProcessor.inverse_normalize()(output_tensor.squeeze(0).cpu())
        output_image = transforms.ToPILImage()(output_tensor.clamp(0, 1))
        output_image.save(self.config.OUTPUT_IMAGE)
        print(f"Stylized image saved to {self.config.OUTPUT_IMAGE}")



In [49]:
command = 'train'  # Change to 'stylize' for stylization

model = StyleTransferModel()
if command == 'train':
    model.train()

Content dataset size: 448, Style dataset size: 50
Starting training...
Epoch [1/200], Step [1/56], Content Loss: 20.9622, Style Loss: 25.1055, Total Loss: 272.0170
Epoch [1/200], Step [2/56], Content Loss: 21.3029, Style Loss: 25.4690, Total Loss: 275.9927
Epoch [1/200], Step [3/56], Content Loss: 20.0463, Style Loss: 23.4729, Total Loss: 254.7756
Epoch [1/200], Step [4/56], Content Loss: 19.1222, Style Loss: 22.5138, Total Loss: 244.2599
Epoch [1/200], Step [5/56], Content Loss: 21.7612, Style Loss: 25.2459, Total Loss: 274.2201
Epoch [1/200], Step [6/56], Content Loss: 17.9067, Style Loss: 20.2365, Total Loss: 220.2717
Saved decoder weights at epoch 1
Epoch [2/200], Step [1/56], Content Loss: 17.4560, Style Loss: 19.9291, Total Loss: 216.7465
Epoch [2/200], Step [2/56], Content Loss: 19.1351, Style Loss: 21.4302, Total Loss: 233.4371
Epoch [2/200], Step [3/56], Content Loss: 19.5229, Style Loss: 20.3194, Total Loss: 222.7170
Epoch [2/200], Step [4/56], Content Loss: 22.0658, Style Lo

In [ ]:
model = StyleTransferModel()
model.stylize()